# Small Language Model (SLM) Fine-Tuning on Google Colab / Kaggle

This notebook fine-tunes a Small Language Model (such as `HuggingFaceTB/SmolLM-135M-Instruct` or `Qwen/Qwen2.5-0.5B-Instruct`) for **Email Intelligence** and **Document Summarization** using **LoRA (Low-Rank Adaptation)** and **PEFT**.

### Recommended Environment:
- Runtime: GPU (T4 on Google Colab or Kaggle P100/T4)
- Memory: 12-16 GB RAM

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q --upgrade transformers datasets peft trl accelerate bitsandbytes

In [ ]:
# Step 3: Imports and Environment Verification
import os
import torch
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 4: Configuration
MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct" # or "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./fine_tuned_slm"
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

In [ ]:
# Step 5: Load Tokenizer and Base Model
print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.config.use_cache = False
print("Base model loaded successfully.")

In [ ]:
# Step 6: Define Instruction Training Dataset
instruction_data = [
    {
        "prompt": "Analyze email and output JSON:",
        "text": "<|im_start|>system\nYou are an executive email intelligence AI. Analyze the email and output ONLY valid JSON with keys: summary, action_items, urgency, sentiment.<|im_end|>\n<|im_start|>user\nFrom: security@corp.com\nSubject: CRITICAL: Security patch required for OpenSSL\nContent: An urgent CVE was announced. All servers must patch OpenSSL by 18:00 UTC today.<|im_end|>\n<|im_start|>assistant\n{\n  \"summary\": \"Emergency security bulletin requiring immediate OpenSSL patching across all servers by 18:00 UTC.\",\n  \"action_items\": [\"Patch OpenSSL on all servers before 18:00 UTC today\"],\n  \"urgency\": \"Critical\",\n  \"sentiment\": \"Negative\"\n}<|im_end|>"
    },
    {
        "prompt": "Analyze email and output JSON:",
        "text": "<|im_start|>system\nYou are an executive email intelligence AI. Analyze the email and output ONLY valid JSON with keys: summary, action_items, urgency, sentiment.<|im_end|>\n<|im_start|>user\nFrom: hr@corp.com\nSubject: Company Picnic Next Month\nContent: We are excited to announce the annual summer picnic on July 14th. Please RSVP in the portal if you plan to attend.<|im_end|>\n<|im_start|>assistant\n{\n  \"summary\": \"Announcement of the annual company picnic on July 14th with an RSVP request.\",\n  \"action_items\": [\"RSVP in the employee portal if attending picnic\"],\n  \"urgency\": \"Low\",\n  \"sentiment\": \"Positive\"\n}<|im_end|>"
    },
    {
        "prompt": "Summarize document section:",
        "text": "<|im_start|>system\nYou are a precise document summarizer. Highlight key points and core decisions concisely.<|im_end|>\n<|im_start|>user\nDocument Section 1 of 2: Small Language Models with parameter counts between 50M and 500M have proven capable of rivaling larger systems on specialized tasks when fine-tuned using targeted instruction datasets.<|im_end|>\n<|im_start|>assistant\nSmall Language Models (50M-500M parameters) effectively match larger models on domain-specific tasks when fine-tuned on focused instruction datasets.<|im_end|>"
    }
]

dataset = Dataset.from_list(instruction_data)
print(f"Prepared {len(dataset)} instruction records.")

In [ ]:
# Step 7: Configure LoRA and Training Arguments
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Step 8: Save Fine-Tuned Weights
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter weights to: {OUTPUT_DIR}")

# Optional: Merge adapter with base model for standalone serving
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=compute_dtype)
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged = peft_model.merge_and_unload()
merged_dir = os.path.join(OUTPUT_DIR, "merged")
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f"Saved merged model to: {merged_dir}")

In [ ]:
# Step 9: Test Inference on a Sample Email
test_email = """<|im_start|>system
You are an executive email intelligence AI. Analyze the email and output ONLY valid JSON with keys: summary, action_items, urgency, sentiment.<|im_end|>
<|im_start|>user
From: devops-alert@startup.io
Subject: Payment Gateway timeout errors spiking above 5%
Content: We are seeing checkout timeouts for the last 15 minutes. Please check Stripe webhook status and redeploy the gateway container immediately.<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_email, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_tokens = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
response_text = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Model Output:")
print(response_text)